# Phase 3. Remove duplicates and build the tracker

**Md Forhadul Islam**. Review of benchmarks for coding agents and agentic software engineering (2022 to 2026).

Every record gets an ID and duplicates are removed: inside OpenAlex (by DOI, then normalised title), between Google
Scholar and OpenAlex (titles without tags such as [PDF]), and for the snowballing records (by permanent OpenAlex ID).

In [ ]:
from setup import *
assert Path.cwd() == ROOT, "the notebook must work inside the Report folder"
print("Report folder:", ROOT)

## 3.1 Google Scholar against OpenAlex

In [ ]:
norm = lambda t: re.sub(r'[^a-z0-9]+', ' ', (t or '').lower()).strip()
one_tag = lambda t: norm(re.sub(r'^\[[A-Z]+\]\s*', '', t or ''))
OA = J('data/openalex_raw_2026-11-09.json'); GS = J('data/scholar_raw_2026-11-09.json')
unique = {}
for r in GS['records']: unique.setdefault(one_tag(r['title']), r)
oa_titles = {norm(r['title']) for r in OA['records']}
new = [r for t, r in unique.items() if t not in oa_titles]
out = {'unique_scholar': len(unique), 'in_openalex': len(unique) - len(new), 'new': len(new), 'new_records': new, 'scholar_totals': GS['log']}
os.makedirs(ROOT / 'search/counts', exist_ok=True)
json.dump(out, open(ROOT / 'search/counts/scholar_overlap_v1.json', 'w'), indent=1)
print('Scholar results', len(GS['records']), '| different titles', len(unique), '| new (first pass)', len(new))

In [ ]:
same_as_original('search/counts/scholar_overlap_v1.json')

## 3.2 IDs for the snowballing records

A snowballing record is new only when its OpenAlex ID is in no earlier set. New records keep the order of the export and get IDs F0001 (forward from the seeds), S0001 (forward from SWE-agent), B0001 (backward) and T0001 (round 2).

In [ ]:
FW = J('data/openalex_forward_snowball_round1_2026-11-09.json'); SA = J('data/openalex_forward_sweagent_and_seeds_2026-11-09.json')
BW = J('data/openalex_backward_snowball_round1_2026-11-09.json'); R2 = J('data/openalex_snowball_round2_2026-11-10.json')
seen = {r['id'] for r in OA['records']}
def new_records(export, prefix, key):
    out = []
    for r in export['records']:
        if r['id'] in seen: continue
        out.append(dict(r, **{key: f'{prefix}{len(out) + 1:04d}'}))
    seen.update(r['id'] for r in export['records'])
    return out
FNEW = new_records(FW, 'F', 'fid'); SNEW = new_records(SA, 'S', 'sid'); BNEW = new_records(BW, 'B', 'bid'); TNEW = new_records(R2, 'T', 'r2id')
for rel, lst in [('snowballing/round_1_forward/forward_new_records_with_ids.json', FNEW), ('snowballing/round_1_forward/sweagent_new_records_with_ids.json', SNEW),
                 ('snowballing/round_1_backward/backward_new_records_with_ids.json', BNEW), ('snowballing/round_2/round2_new_records_with_ids.json', TNEW)]:
    os.makedirs((ROOT / rel).parent, exist_ok=True); json.dump(lst, open(ROOT / rel, 'w'), indent=1); print(rel, len(lst))
expect('new forward records', len(FNEW), 2982); expect('new SWE-agent records', len(SNEW), 780)
expect('new backward records', len(BNEW), 5965); expect('new round 2 records', len(TNEW), 4717)

In [ ]:
for rel in ['snowballing/round_1_forward/forward_new_records_with_ids.json', 'snowballing/round_1_forward/sweagent_new_records_with_ids.json',
            'snowballing/round_1_backward/backward_new_records_with_ids.json', 'snowballing/round_2/round2_new_records_with_ids.json']: same_as_original(rel)

## 3.3 The first tracker and the duplicates inside OpenAlex

### `scripts/01_build_tracker.py`

Removes duplicates inside OpenAlex (DOI, then normalised title), numbers the records R0001 onward and builds the first tracker.

In [ ]:
%%writefile scripts/01_build_tracker.py
"""Build master tracker from raw OpenAlex export. All counts come from this script, never typed by hand."""
import json,re,sys,os,collections
from openpyxl import Workbook
from openpyxl.styles import Font,Alignment,PatternFill
root=sys.argv[1]
raw=json.load(open(os.path.join(root,'data','openalex_raw_2026-11-09.json'),encoding='utf-8'))
recs=raw['records']
def norm(t): return re.sub(r'[^a-z0-9]+',' ',(t or '').lower()).strip()
# dedup: DOI then normalised title
seen_doi={};seen_t={};uniq=[];dups=[]
for r in recs:
    d=(r['doi'] or '').lower().replace('https://doi.org/','')
    t=norm(r['title'])
    if (d and d in seen_doi) or (t and t in seen_t):
        dups.append(r['id']);continue
    if d: seen_doi[d]=r['id']
    if t: seen_t[t]=r['id']
    uniq.append(r)
# priority flag for stage 1 (aid only; human decides)
BEN=re.compile(r'\b(benchmark|benchmarks|bench|dataset|datasets|leaderboard)\b',re.I)
AGT=re.compile(r'(swe-?bench|coding agent|software engineering agent|code agent|agentic|llm agent|repository[- ]level|issue resolution|issue[- ]resolving|program repair|code generation|software engineering)',re.I)
SUR=re.compile(r'\b(survey|systematic (literature )?review|literature review|mapping study|taxonomy)\b',re.I)
CON=re.compile(r'contaminat|data leak|memoriz',re.I)
def flag(r):
    txt=(r['title'] or '')+' '+(r['abstract'] or '')
    if not r['abstract']: return 'C_no_abstract'
    b=bool(BEN.search(txt));a=bool(AGT.search(txt))
    tb=bool(BEN.search(r['title'] or '')) 
    if SUR.search(r['title'] or '') and a: return 'S_survey_or_review'
    if b and a and tb: return 'A_likely_benchmark_paper'
    if b and a: return 'B_possible'
    return 'D_unlikely'
cnt=collections.Counter()
rows=[]
for i,r in enumerate(sorted(uniq,key=lambda x:(-x['cites'] if x['cites'] else 0)),1):
    f=flag(r);cnt[f]+=1
    rows.append((f'R{i:04d}',r,f))
wb=Workbook();ws=wb.active;ws.title='Master_Tracker'
hdr=['ID','Title','Authors (first 6)','Year','Date','Venue','Type','Cited by','DOI','Open access','OA / PDF link','Found via queries','Auto priority (aid only)','Contamination words in abstract','Stage 1 decision (Include/Exclude/Maybe)','Stage 1 reason','Stage 2 decision','Exclusion code','Full text saved (path)','Snowball round','Benchmark name(s)','Notes','Abstract','OpenAlex ID']
ws.append(hdr)
for c in ws[1]:
    c.font=Font(bold=True,color='FFFFFF');c.fill=PatternFill('solid',fgColor='1F3864');c.alignment=Alignment(wrap_text=True,vertical='top')
for rid,r,f in rows:
    ws.append([rid,r['title'],r['authors'],r['year'],r['date'],r['venue'],r['type'],r['cites'],r['doi'],'yes' if r['oa'] else 'no',r['oa_url'],','.join(r['queries']),f,'yes' if CON.search(r['abstract'] or '') else '','','','','','','','','',r['abstract'],r['id']])
for col,w in zip('ABCDEFGHIJKLMNOPQRSTUVWX',[8,60,30,6,11,28,12,8,28,8,40,10,22,10,16,20,14,12,24,8,20,24,90,28]): ws.column_dimensions[col].width=w
ws.freeze_panes='C2';ws.auto_filter.ref=ws.dimensions
# counts sheet
cs=wb.create_sheet('PRISMA_Counts')
by_year=collections.Counter(r['year'] for _,r,_ in rows)
q=collections.Counter()
for _,r,_ in rows:
    for k in r['queries']: q[k]+=1
log=raw['log']
counts={'records_identified_total_rows':len(recs),'duplicates_removed':len(dups),'unique_records':len(uniq),'with_abstract':sum(1 for _,r,_ in rows if r['abstract']),'open_access':sum(1 for _,r,_ in rows if r['oa']),'auto_priority':dict(cnt),'by_year':dict(sorted(by_year.items())),'by_query_unique':dict(q),'query_log':log,'note':'Auto priority is a sorting aid. Stage 1 decisions are made by the reviewer, not by this script.'}
for k,v in [('Records identified (sum of query hits fetched, before dedup)',len(recs)),('Duplicates removed',len(dups)),('Unique records',len(uniq))]: cs.append([k,v])
cs.append([]);cs.append(['Auto priority bucket','Count'])
for k,v in sorted(cnt.items()): cs.append([k,v])
cs.append([]);cs.append(['Year','Unique records'])
for k,v in sorted(by_year.items()): cs.append([k,v])
cs.column_dimensions['A'].width=60
os.makedirs(os.path.join(root,'tracker','master_tracker'),exist_ok=True)
wb.save(os.path.join(root,'tracker','master_tracker','Master_Tracker_v1.xlsx'))
json.dump(counts,open(os.path.join(root,'search','counts','prisma_counts_v1.json'),'w'),indent=1)
print(json.dumps({k:v for k,v in counts.items() if k!='query_log'},indent=1))

In [ ]:
p = run('01_build_tracker.py')
c = J('search/counts/prisma_counts_v1.json'); expect('OpenAlex duplicates', c['duplicates_removed'], 117); expect('unique OpenAlex records', c['unique_records'], 3247)
same_as_original('search/counts/prisma_counts_v1.json')

### `scripts/02_add_scholar_sheet.py`

Adds the Google Scholar records not found in OpenAlex to the tracker (IDs GS001 onward).

In [ ]:
%%writefile scripts/02_add_scholar_sheet.py
"""Add Google Scholar only records (not found in OpenAlex) to the tracker as sheet Scholar_Only. Snippet is partial abstract; full abstract to be filled when paper is read."""
import json,sys,openpyxl
from openpyxl.styles import Font,PatternFill,Alignment
R=sys.argv[1]
d=json.load(open(R+'/search/counts/scholar_overlap_v1.json',encoding='utf-8'))
p=R+'/tracker/master_tracker/Master_Tracker_v1.xlsx'
wb=openpyxl.load_workbook(p)
if 'Scholar_Only' in wb.sheetnames: del wb['Scholar_Only']
ws=wb.create_sheet('Scholar_Only')
hdr=['ID','Title','Authors, venue, year (Scholar line)','Cited by','URL','PDF link','Found via Scholar query','Snippet (partial abstract)','Stage 1 decision','Reason','Stage 2 decision','Full text saved (path)','Full abstract (fill when read)','Notes']
ws.append(hdr)
for c in ws[1]: c.font=Font(bold=True,color='FFFFFF');c.fill=PatternFill('solid',fgColor='7B3F00');c.alignment=Alignment(wrap_text=True)
for i,r in enumerate(d['new_records'],1):
    ws.append([f'GS{i:03d}',r['title'],r['meta'],r['cited_by'],r['url'],r['pdf'],r['q'],r['snippet'],'','','','','',''])
for col,w in zip('ABCDEFGHIJKLMN',[8,60,50,8,40,40,10,90,12,20,12,24,60,24]): ws.column_dimensions[col].width=w
cs=wb['PRISMA_Counts']
cs.append([]);cs.append(['Google Scholar (first 50 per query, since 2022)','Count'])
for l in d['scholar_totals']:
    cs.append([l['k']+': '+(l.get('query') or ''),l.get('scholar_total_text','')])
cs.append(['Unique Scholar records',d['unique_scholar']]);cs.append(['Already in OpenAlex set',d['in_openalex']]);cs.append(['New from Scholar (sheet Scholar_Only)',d['new']])
wb.save(p);print('ok',d['new'])

In [ ]:
p = run('02_add_scholar_sheet.py')

### `scripts/02b_scholar_dedupe_fix.py`

Removes every leading tag (some titles carry two, such as [PDF][PDF]) and finds five more Scholar duplicates.

In [ ]:
%%writefile scripts/02b_scholar_dedupe_fix.py
"""Fix found by the proof notebook: strip repeated tags like [PDF][PDF] before matching Scholar titles to OpenAlex.
Writes search/counts/scholar_overlap_v2.json with the GS ids that are duplicates of OpenAlex records."""
import json,re,sys
R=sys.argv[1]
def norm(t):
    t=re.sub(r'^(\s*\[[A-Z]+\]\s*)+','',t or '')
    return re.sub(r'[^a-z0-9]+',' ',t.lower()).strip()
ot={norm(r['title']) for r in json.load(open(R+'/data/openalex_raw_2026-11-09.json',encoding='utf-8'))['records']}
v1=json.load(open(R+'/search/counts/scholar_overlap_v1.json'))
dups=[];keep=[]
for k,r in enumerate(v1['new_records'],1):
    gid=f'GS{k:03d}'
    (dups if norm(r['title']) in ot else keep).append(gid)
dec={d['id']:d['decision'] for d in json.load(open(R+'/screening/stage1_title_abstract/decisions/scholar_only_decisions.json'))}
out={'unique_scholar':v1['unique_scholar'],'in_openalex':v1['unique_scholar']-len(keep),'new':len(keep),'duplicate_gs_ids':dups,'duplicate_stage1_decisions':{g:dec.get(g) for g in dups},'note':'repeated bracket tags are stripped before title matching'}
json.dump(out,open(R+'/search/counts/scholar_overlap_v2.json','w'),indent=1);print(out)

In [ ]:
p = run('02b_scholar_dedupe_fix.py')
same_as_original('search/counts/scholar_overlap_v2.json')

## 3.4 One readable check of the whole search and deduplication

In [ ]:
os.makedirs(ROOT / 'plan/scripts', exist_ok=True)

### `plan/scripts/search_dedupe_snowball.py`

Re-counts the saved exports and checks each result for the database search, the first snowballing round and round 2 (steps 1, 4 and 5 re-count the saved exports, steps 2, 3 and 6 rebuild the duplicate removal; the search functions are shown but not run in check mode). Rounds 3 to 5 are not covered here (see notebook 07). Any failed internal check makes the script, and so this notebook check, fail.

In [ ]:
%%writefile plan/scripts/search_dedupe_snowball.py
"""Search, deduplication and snowballing (database search, round 1 and round 2) for the coding agent benchmark review, in one readable file.

Author: Md Forhadul Islam.

Usage:
    python3 search_dedupe_snowball.py <review_root>          # check mode: rebuilds every step from the saved
                                                             # raw files and compares with the saved results
    python3 search_dedupe_snowball.py <review_root> --live   # also shows how the live OpenAlex calls are made
                                                             # (a live run today returns newer records)

In check mode the steps that re-count saved exports are steps 1, 4 and 5 (they print the saved counts) and steps 2, 3 and 6
(they rebuild the duplicate removal and compare it with the saved results). The functions search_openalex(), forward() and
backward() are shown so that the calls can be read, but they are not run in check mode, so the searches themselves are not
rebuilt here (notebooks 02 and 07 repeat them live). Snowballing rounds 3 to 5 are not covered by this file (see notebook 07).

The search itself was run on 9 and 10 November 2026 in my Chrome browser (the JavaScript versions are in
scripts/browser_js, written by notebook 02). The functions below are the same calls written in Python, so the logic can be read
in one place. Steps:
    1. OpenAlex search (5 query blocks)                       -> data/openalex_raw_2026-11-09.json
    2. Remove duplicates inside the OpenAlex results (DOI, then title)
    3. Google Scholar results: remove duplicates and the ones already in OpenAlex
    4. Forward snowballing: papers that cite each seed benchmark
    5. Backward snowballing: papers cited by the papers kept at stage 1
    6. Snowballing round 2: both directions from the papers newly included in round 1
    7. Summary table of identified, duplicate and new records
"""
import json, os, re, sys, time, urllib.request, urllib.parse

R = sys.argv[1] if len(sys.argv) > 1 and not sys.argv[1].startswith('--') else '.'
LIVE = '--live' in sys.argv
J = lambda p: json.load(open(os.path.join(R, p), encoding='utf-8'))
OK = []
def check(name, cond, detail=''):
    OK.append(bool(cond)); print(('PASS  ' if cond else 'FAIL  ') + name + (f'   [{detail}]' if detail != '' else ''))

# ------------------------------------------------------------------------------------------------
# Shared settings
# ------------------------------------------------------------------------------------------------
API = 'https://api.openalex.org/works'
DATE_FROM, DATE_TO = '2022-11-01', '2026-11-09'
FIELDS = ('id,doi,title,publication_year,publication_date,type,cited_by_count,primary_location,'
          'best_oa_location,open_access,authorships,abstract_inverted_index,referenced_works')
QUERIES = {
    'A': '("coding agent" OR "coding agents" OR "software engineering agent" OR "code agent") AND (benchmark OR dataset)',
    'B': '("SWE-bench" OR "repository-level code" OR "issue resolution") AND (benchmark OR dataset)',
    'C': '("agentic" AND "software engineering") AND (benchmark OR evaluation)',
    'D': '("data contamination" OR "benchmark contamination") AND (code OR "software engineering")',
    'E': '("coding agents" OR "code generation") AND benchmark AND (survey OR "systematic review")'}

def norm(title):
    """Normalise a title for matching: lower case, letters and digits only, single spaces."""
    return re.sub(r'[^a-z0-9]+', ' ', (title or '').lower()).strip()

def abstract_from_index(inv):
    """OpenAlex stores abstracts as {word: [positions]}; rebuild the plain text."""
    if not inv: return ''
    words = {}
    for w, pos in inv.items():
        for p in pos: words[p] = w
    return ' '.join(words[i] for i in sorted(words))

def get(url):
    with urllib.request.urlopen(url, timeout=60) as r: return json.loads(r.read())

def openalex_pages(filter_text):
    """All results for one OpenAlex filter, 200 per page. Page paging is used because cursor paging needs a key."""
    page, out = 1, []
    while True:
        j = get(f'{API}?filter={filter_text}&per-page=200&select={FIELDS}&page={page}')
        out += j['results']
        if page * 200 >= j['meta']['count']: return out, j['meta']['count']
        page += 1; time.sleep(0.2)

# ------------------------------------------------------------------------------------------------
# Step 1. OpenAlex search
# ------------------------------------------------------------------------------------------------
def search_openalex():
    """Title and abstract search for each query block, limited to the review window."""
    records, log = {}, []
    for k, q in QUERIES.items():
        f = f'title_and_abstract.search:{urllib.parse.quote(q)},from_publication_date:{DATE_FROM},to_publication_date:{DATE_TO}'
        works, total = openalex_pages(f)
        for w in works:
            r = records.setdefault(w['id'], {'id': w['id'], 'doi': w.get('doi'), 'title': w.get('title'),
                                             'year': w.get('publication_year'), 'abstract': abstract_from_index(w.get('abstract_inverted_index')),
                                             'refs': '|'.join(w.get('referenced_works') or []), 'queries': []})
            r['queries'].append(k)
        log.append({'k': k, 'query': q, 'total_hits': total, 'fetched': len(works)})
    return list(records.values()), log

print('STEP 1. OpenAlex search')
OA = J('data/openalex_raw_2026-11-09.json')
for l in OA['log']: print(f"  query {l['k']}: {l.get('total_hits')} hits")
print('  records saved (one per OpenAlex work):', len(OA['records']))
c1 = J('search/counts/prisma_counts_v1.json')
check('OpenAlex records equal the saved total', len(OA['records']) == c1['records_identified_total_rows'], len(OA['records']))
if LIVE:
    print('  live run of query A only, to show the call works:')
    f = f"title_and_abstract.search:{urllib.parse.quote(QUERIES['A'])},from_publication_date:{DATE_FROM},to_publication_date:{DATE_TO}"
    print('  ', get(f'{API}?filter={f}&per-page=1')['meta']['count'], 'hits today')

# ------------------------------------------------------------------------------------------------
# Step 2. Duplicates inside the OpenAlex results
# A record is a duplicate when its DOI, or its normalised title, has already been seen.
# ------------------------------------------------------------------------------------------------
def dedupe(records):
    seen_doi, seen_title, unique, dups = set(), set(), [], []
    for r in records:
        d = (r.get('doi') or '').lower().replace('https://doi.org/', ''); t = norm(r.get('title'))
        if (d and d in seen_doi) or (t and t in seen_title):
            dups.append(r['id']); continue
        if d: seen_doi.add(d)
        if t: seen_title.add(t)
        unique.append(r)
    return unique, dups

print('\nSTEP 2. Remove duplicates inside OpenAlex (DOI, then normalised title)')
oa_unique, oa_dups = dedupe(OA['records'])
print(f'  {len(OA["records"])} records, {len(oa_dups)} duplicates, {len(oa_unique)} unique')
check('OpenAlex duplicates equal the saved count', len(oa_dups) == c1['duplicates_removed'], len(oa_dups))

# ------------------------------------------------------------------------------------------------
# Step 3. Google Scholar
# Scholar titles start with tags such as [PDF] or [HTML]; remove every leading tag before matching.
# Keep one result per title, then drop the titles already found in OpenAlex.
# ------------------------------------------------------------------------------------------------
print('\nSTEP 3. Google Scholar: remove repeats and records already in OpenAlex')
GS = J('data/scholar_raw_2026-11-09.json')
strip_tags = lambda t: re.sub(r'^(\s*\[[A-Z]+\]\s*)+', '', t or '')
gs_unique = {}
for r in GS['records']: gs_unique.setdefault(norm(strip_tags(r['title'])), r)
oa_titles = {norm(r['title']) for r in OA['records']}
gs_new = [t for t in gs_unique if t not in oa_titles]
v2 = J('search/counts/scholar_overlap_v2.json')
print(f'  {len(GS["records"])} Scholar results, {len(gs_unique)} different titles, {len(gs_unique) - len(gs_new)} already in OpenAlex, {len(gs_new)} new')
check('Scholar new records equal the saved count', len(gs_new) == v2['new'], f'{len(gs_new)} vs {v2["new"]}')

# ------------------------------------------------------------------------------------------------
# Step 4. Forward snowballing: every work that cites a seed paper (OpenAlex filter cites:W...)
# ------------------------------------------------------------------------------------------------
def forward(seed_id):
    works, total = openalex_pages(f'cites:{seed_id},from_publication_date:{DATE_FROM}')
    return works

print('\nSTEP 4. Forward snowballing from the seed benchmarks, then from SWE-agent')
FW = J('data/openalex_forward_snowball_round1_2026-11-09.json')
SA = J('data/openalex_forward_sweagent_and_seeds_2026-11-09.json')
for l in FW['log']: print(f"  {l.get('seed', l.get('label', ''))}: {l.get('forward_citing_total')} citing works")
oa_ids = {r['id'] for r in OA['records']}
f_new = [r for r in FW['records'] if r['id'] not in oa_ids]                       # new = not already found
s_new = [r for r in SA['records'] if r['id'] not in oa_ids | {r['id'] for r in FW['records']}]
FNEW = J('snowballing/round_1_forward/forward_new_records_with_ids.json')
SNEW = J('snowballing/round_1_forward/sweagent_new_records_with_ids.json')
print(f'  seeds: {len(FW["records"])} citing works, {len(f_new)} new | SWE-agent: {len(SA["records"])} citing works, {len(s_new)} new')
check('Forward new records equal the saved list', {r['id'] for r in f_new} == {r['id'] for r in FNEW}, len(f_new))
check('SWE-agent new records equal the saved list', {r['id'] for r in s_new} == {r['id'] for r in SNEW}, len(s_new))

# ------------------------------------------------------------------------------------------------
# Step 5. Backward snowballing: every work cited by the papers kept at stage 1
# (OpenAlex filter cited_by:W1|W2|..., 40 ids per call)
# ------------------------------------------------------------------------------------------------
def backward(paper_ids):
    found = {}
    for i in range(0, len(paper_ids), 40):
        works, _ = openalex_pages(f'cited_by:{"|".join(paper_ids[i:i + 40])},from_publication_date:{DATE_FROM}')
        for w in works: found.setdefault(w['id'], w)
    return list(found.values())

print('\nSTEP 5. Backward snowballing from the papers kept at stage 1')
BW = J('data/openalex_backward_snowball_round1_2026-11-09.json')
earlier = oa_ids | {r['id'] for r in FW['records']} | {r['id'] for r in SA['records']}
b_new = [r for r in BW['records'] if r['id'] not in earlier]
BNEW = J('snowballing/round_1_backward/backward_new_records_with_ids.json')
print(f'  {len(BW["records"])} cited works, {len(b_new)} new')
check('Backward new records equal the saved list', {r['id'] for r in b_new} == {r['id'] for r in BNEW}, len(b_new))

# ------------------------------------------------------------------------------------------------
# Step 6. Round 2: both directions from the papers newly included in round 1
# ------------------------------------------------------------------------------------------------
print('\nSTEP 6. Snowballing round 2')
R2 = J('snowballing/round_2/openalex_snowball_round2_2026-11-10.json')
seeds2 = [s for s in open(os.path.join(R, 'snowballing/round_2_seed_ids.txt')).read().replace(',', ' ').split() if s]
earlier2 = earlier | {r['id'] for r in BW['records']}
t_new = [r for r in R2['records'] if r['id'] not in earlier2]
TNEW = J('snowballing/round_2/round2_new_records_with_ids.json')
print(f'  start set {len(seeds2)} papers, {len(R2["records"])} works found, {len(t_new)} new')
check('Round 2 new records equal the saved list', {r['id'] for r in t_new} == {r['id'] for r in TNEW}, len(t_new))

# ------------------------------------------------------------------------------------------------
# Step 7. Summary
# ------------------------------------------------------------------------------------------------
print('\nSTEP 7. Summary')
rows = [('OpenAlex search', len(OA['records']), len(oa_dups), len(oa_unique)),
        ('Google Scholar', len(GS['records']), len(GS['records']) - len(gs_new), len(gs_new)),
        ('Forward, seeds', len(FW['records']), len(FW['records']) - len(f_new), len(f_new)),
        ('Forward, SWE-agent', len(SA['records']), len(SA['records']) - len(s_new), len(s_new)),
        ('Backward, round 1', len(BW['records']), len(BW['records']) - len(b_new), len(b_new)),
        ('Round 2', len(R2['records']), len(R2['records']) - len(t_new), len(t_new))]
print(f'  {"source":22} {"found":>8} {"duplicate or seen":>18} {"new to screen":>14}')
for s, a, b, c in rows: print(f'  {s:22} {a:8,} {b:18,} {c:14,}')
print(f'  {"total":22} {sum(r[1] for r in rows):8,} {sum(r[2] for r in rows):18,} {sum(r[3] for r in rows):14,}')
print(f'\n{sum(OK)} of {len(OK)} checks pass')
sys.exit(0 if all(OK) else 1)

In [ ]:
p = run('search_dedupe_snowball.py', folder='plan/scripts')

In [ ]:
save_checks('03_Deduplicate_and_tracker')